# Phase 2: Preprocessing, Data Cleaning, and Operational Categorization
## Flight Delays Are Contagious: Predicting Delay Propagation Across Consecutive Aircraft Operations

**Objective:** Build a standardized, leak-free, clean, and enriched base flight dataset from all 12 monthly BTS 2025 datasets without modifying raw files or prematurely reconstructing rotations.

**Core Pillars of Phase 2:**
1. **Operational Categorization:** Segregate flights into Category A (Completed & Valid), Category B (Cancelled), Category C (Diverted), and Category D (Invalid/Incomplete).
2. **Airport Geolocation & IANA Timezone Resolution:** Resolve 100% of the 352 active airports to coordinates, official IANA time zones, and DST observation status.
3. **Local Timestamp Deconstruction:** Parse 4-digit `HHMM` fields into validated hours, minutes, minutes-of-day, local departure datetimes, and flags for midnight boundaries, overnight operations, and DST fallback ambiguities.
4. **Missing Value Preservation:** Maintain structural missingness (delay causes, unobserved actual times) in compliance with BTS DOT reporting regulations.
5. **High-Performance Storage:** Export clean datasets to compressed Parquet (`flights_cleaned.parquet`, `cancelled_flights.parquet`, `diverted_flights.parquet`).

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

con = duckdb.connect()
print("Environment initialized successfully.")

### 1. Operational Flight Categorization Summary

In [ ]:
df_prep_summary = pd.read_csv('../results/tables/preprocessing_summary.csv')
print(f"Total Input Records: {df_prep_summary['record_count'].sum():,}")
df_prep_summary

### 2. Airport Metadata & IANA Timezone Enrichment

In [ ]:
df_airports = pd.read_csv('../results/tables/airport_enrichment_validation.csv')
print(f"Total active airports enriched: {len(df_airports)}")
print("Distribution of IANA Timezones:")
print(df_airports['IANA_TIMEZONE'].value_counts())

# Sample preview
df_airports[['AIRPORT_ID', 'AIRPORT', 'DISPLAY_AIRPORT_NAME', 'AIRPORT_STATE_CODE', 'LATITUDE', 'LONGITUDE', 'IANA_TIMEZONE', 'OBSERVES_DST']].head(10)

### 3. Local Timestamp Deconstruction & Operational Boundaries

In [ ]:
df_ts_val = pd.read_csv('../results/tables/timestamp_validation_report.csv')
df_ts_val

### 4. Structural Missing-Value Treatment & Semantics

In [ ]:
df_mvt = pd.read_csv('../results/tables/missing_value_treatment.csv')
# Show columns with raw nulls to verify their specific treatment policy
null_cols = df_mvt[df_mvt['raw_null_count'] > 0]
null_cols[['column_name', 'raw_null_count', 'raw_null_pct', 'category_a_treatment', 'semantic_reason_and_policy']]

### 5. Verification of Exported Parquet Datasets

In [ ]:
cleaned_path = '../data/processed/flights_cleaned.parquet'
cancelled_path = '../data/processed/cancelled_flights.parquet'
diverted_path = '../data/processed/diverted_flights.parquet'

count_a = con.execute(f"SELECT COUNT(*) FROM read_parquet('{cleaned_path}')").fetchone()[0]
count_b = con.execute(f"SELECT COUNT(*) FROM read_parquet('{cancelled_path}')").fetchone()[0]
count_c = con.execute(f"SELECT COUNT(*) FROM read_parquet('{diverted_path}')").fetchone()[0]

size_a_mb = os.path.getsize(cleaned_path) / (1024 * 1024)
size_b_mb = os.path.getsize(cancelled_path) / (1024 * 1024)
size_c_mb = os.path.getsize(diverted_path) / (1024 * 1024)

df_parquet_stats = pd.DataFrame([
    {'Dataset': 'Category A: flights_cleaned.parquet', 'Rows': count_a, 'Size_MB': round(size_a_mb, 2)},
    {'Dataset': 'Category B: cancelled_flights.parquet', 'Rows': count_b, 'Size_MB': round(size_b_mb, 2)},
    {'Dataset': 'Category C: diverted_flights.parquet', 'Rows': count_c, 'Size_MB': round(size_c_mb, 2)}
])
df_parquet_stats

### 6. Sample Records from Cleaned Base Dataset

In [ ]:
sample_cols = [
    'FL_DATE', 'OP_UNIQUE_CARRIER', 'TAIL_NUM', 'OP_CARRIER_FL_NUM',
    'ORIGIN', 'ORIGIN_STATE', 'DEST', 'DEST_STATE',
    'CRS_DEP_TIME', 'CRS_DEP_LOCAL_DATETIME', 'DEP_TIME', 'DEP_DELAY',
    'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DEL15',
    'ORIGIN_IANA_TZ', 'DEST_IANA_TZ', 'IS_SCHEDULED_OVERNIGHT', 'IS_DEP_DELAYED_PAST_MIDNIGHT'
]
df_sample = con.execute(f"SELECT {', '.join(sample_cols)} FROM read_parquet('{cleaned_path}') LIMIT 10").df()
df_sample

### 7. Phase 2 Conclusion and Transition to Phase 3

**Summary of Deliverables:**
1. Ingestion of 7,001,619 raw BTS flight records successfully completed and partitioned.
2. 6,879,483 valid completed flight records (Category A) saved to `data/processed/flights_cleaned.parquet` (356.68 MB).
3. 102,876 cancelled flights and 19,258 diverted flights saved to separate dedicated datasets.
4. Exactly 2 anomalous records (Category D) identified, documented, and safely isolated.
5. 100% of 352 active airports geolocated and mapped to official IANA time zones with DST rules.
6. Timestamp validation confirmed zero parsing failures; midnight `2400` notation, overnight operations, and DST ambiguity on 2025-11-02 were explicitly resolved or flagged.

**Ready for Phase 3:** Timestamp normalization to UTC and aircraft rotation reconstruction (`Previous Flight -> Next Flight`).